## Import Essential Libraries

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

## Import the DataSet

In [2]:
data = pd.read_csv('Data.csv')

## Checking the information of the DataSet

In [27]:
data.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [5]:
data.shape

(8790, 10)

In [6]:
data.describe()

,release_year
count,8790.000000
mean,2014.183163
std,8.825466
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


## Removing Unnecessary Columns

In [31]:
data.drop(['show_id', 'date_added', 'release_year'], axis = 1, inplace = True)

## Checking Null Values in DataSet

In [8]:
data.isna().sum()

type         0
title        0
director     0
country      0
rating       0
duration     0
listed_in    0
dtype: int64

## Numeric Encoding

In [32]:
data['duration'] = data['duration'].astype(str).str.extract(r'(\d+)').astype(str)

## Replace Space from _

In [33]:
for i in data.columns:
    df = []
    for j in data[i]:
        df.append(j.replace(' ', '_'))
    data[i] = df 

## Merge the Colume 

In [11]:
data['tag'] = ''
for i in data.columns:
    data['tag'] += data[i].astype(str) + ' '

## Seprating Independent Variable

In [12]:
X = data['tag']

## Encode the DataSet

### Tokenizer

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
token = Tokenizer(num_words = 1000, oov_token = "<OOV>")
token.fit_on_texts(X)
X = token.texts_to_sequences(X)

### Padding

In [14]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X = pad_sequences(X, maxlen = 100, padding = "post", truncating = "post")
print("Input shape:", X.shape)

Input shape: (8790, 100)


## Normalize

In [15]:
X = X / X.max()

### Create CNN Model

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, Dense, Dropout, Embedding, GlobalMaxPooling1D, GlobalAveragePooling1D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [17]:
K.clear_session()
model = Sequential()
model.add(Input(shape=(100,)))
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(64, activation='relu', name='movie_embedding'))
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(100, activation='sigmoid'))

## Compile Model

In [18]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mae',
    metrics=['mae']
)

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        51,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ movie_embedding (Dense)         │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 100)            │        51,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 455,588 (1.74 MB)

 Trainable params: 452,004 (1.72 MB)

 Non-trainable params: 3,584 (14.00 KB)

## Training tht Model

In [20]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
early_stop = EarlyStopping(monitor='val_mae', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_mae', patience=2, factor=0.3)
history = model.fit(X, X, validation_split = 0.2, epochs=50, batch_size=32, callbacks=[early_stop, reduce_lr])

Epoch 1/50


220/220 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - loss: 0.1364 - mae: 0.1364 - val_loss: 0.0146 - val_mae: 0.0146 - learning_rate: 0.0010
Epoch 2/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0584 - mae: 0.0584 - val_loss: 0.0149 - val_mae: 0.0149 - learning_rate: 0.0010
Epoch 3/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0270 - mae: 0.0270 - val_loss: 0.0137 - val_mae: 0.0137 - learning_rate: 0.0010
Epoch 4/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0186 - mae: 0.0186 - val_loss: 0.0136 - val_mae: 0.0136 - learning_rate: 0.0010
Epoch 5/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0181 - mae: 0.0181 - val_loss: 0.0129 - val_mae: 0.0129 - learning_rate: 0.0010
Epoch 6/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0177 - mae: 0.0177 - val_loss: 0.0121 - val_mae: 0.0121 - learning_rate: 0.0010
Epoch 7/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0171 - mae: 0.0171 - val_loss: 0.0122 - val_mae: 0.0122 - learning_rate: 0.0010
Epoch 8/5

## Embedding the Model

In [21]:
embedding_model = Model(
    inputs=model.inputs,
    outputs=model.get_layer("movie_embedding").output
)
movie_embeddings = embedding_model.predict(X)
print("Movie Embedding Shape:", movie_embeddings)

275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Movie Embedding Shape: [[1.1474917  0.         0.         ... 0.         0.         1.1771064 ]
 [0.         0.         0.10959572 ... 0.         0.         0.37026232]
 [0.3038391  0.         0.         ... 0.         0.         0.        ]
 ...
 [0.73049885 0.         0.         ... 0.39594087 0.         1.4067764 ]
 [0.23064652 0.         0.         ... 0.         0.         0.10883336]
 [0.22963652 0.         0.         ... 0.         0.         0.10421021]]


## Creating the Similarity Matrix

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(movie_embeddings)

## Saving the Similarity Matrix

In [23]:
import joblib
joblib.dump(similarity_matrix, 'similarity_matrix.pkl')

['similarity_matrix.pkl']